In [1]:
"""
The purpose of this Jupyter notebook is to estimate the PU class prior
via the method of Elkan and Noto.

To this end, a logistic regression model is used as a cheap proxy model.
"""

'\nThe purpose of this Jupyter notebook is to estimate the PU class prior\nvia the method of Elkan and Noto.\n\nTo this end, a logistic regression model is used as a cheap proxy model.\n'

In [1]:
import ast
import pickle

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

# Define Functions for Prior Estimation

In [2]:
def estimate_prior_elkan_noto_cv(
    pheno,
    ppi,
    labels,
    n_splits=5,
    random_state=42,
    verbose=True
):
    """
    Cross-validated Elkan & Noto prior estimation.

    Parameters
    ----------
    pheno : np.ndarray (N, 3)
    ppi : np.ndarray (N, 440)
    labels : np.ndarray (N,)
        1 = known positive, 0 = unlabeled
    n_splits : int
    random_state : int

    Returns
    -------
    pi : float
        Estimated class prior
    """

    X = np.concatenate([pheno, ppi], axis=1)
    y = labels

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    pos_probs_all = []
    all_probs_all = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]

        # Train proxy classifier
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_train, y_train)

        probs_val = clf.predict_proba(X_val)[:, 1]

        # Collect probabilities
        pos_probs = probs_val[y_val == 1]

        pos_probs_all.append(pos_probs)
        all_probs_all.append(probs_val)

        if verbose:
            print(f"Fold {fold}: "
                  f"pos_mean={pos_probs.mean():.4f}, "
                  f"all_mean={probs_val.mean():.4f}")

    pos_probs_all = np.concatenate(pos_probs_all)
    all_probs_all = np.concatenate(all_probs_all)

    # Estimate c and π
    c = pos_probs_all.mean()
    pi = all_probs_all.mean() / c

    # Stability safeguards
    pi = np.clip(pi, 1e-3, 0.9)

    if verbose:
        print("\nFinal estimates:")
        print(f"c (P(s=1|y=1)) = {c:.6f}")
        print(f"π (class prior) = {pi:.6f}")

    return pi


def estimate_prior_elkan_noto_repeated(
    pheno,
    ppi,
    labels,
    n_repeats=5,
    **kwargs
):
    pis = []

    for i in range(n_repeats):
        pi = estimate_prior_elkan_noto_cv(
            pheno,
            ppi,
            labels,
            random_state=42 + i,
            verbose=False,
            **kwargs
        )
        pis.append(pi)

    pis = np.array(pis)

    print(f"π mean = {pis.mean():.4f}, std = {pis.std():.4f}")

    return pis.mean()

# Load the Data Required for Prior Estimation

## Load and Prepare the PPI Probability Data

In [3]:
# Also load the predicted PPI probabilities
ppi_preds_path = (
    "/Users/jacobanter/Documents/Code/VACV_screen/Positive_unlabeled_"
    "learning/Inference_on_Dharmacon_pooled_G1_G2_screening_plates_"
    "subset/all_ppi_predictions.tsv"
)

ppi_preds_df = pd.read_csv(
    ppi_preds_path,
    sep="\t"
)

In [4]:
# Create a dictionary mapping UniProt accessions to their corresponding
# PPI probability vector of length 440
prot_to_ppi_prob_vector_dict = {}

# To this end, the `.groupby()` method is employed
for protein, group in ppi_preds_df.groupby(by="protein_1"):
    ppi_prob_vector = group["interaction_probability"].to_numpy()
    assert ppi_prob_vector.shape[0] == 440, (
        f"For protein {protein}, some PPI probabilities are missing!"
    )
    prot_to_ppi_prob_vector_dict[protein] = ppi_prob_vector

## Load and Prepare the PU Training Set

In [6]:
# For prior estimation, the trainig set of the PU data set is used
pu_training_set_path = (
    "/Users/jacobanter/Documents/Code/VACV_screen/Positive_unlabeled_"
    "learning/Train_validation_test_split/PU_data_set_splits_PU_ratio_"
    "1_5/PU_training_set.tsv"
)

pu_training_set_df = pd.read_csv(
    pu_training_set_path,
    sep="\t",
    converters={
        "phenotype_vec": ast.literal_eval,
        "protein_ids": ast.literal_eval
    }
)

In [7]:
# The probabilities must first be aggregated as some genes encode
# multiple proteins
# Create a dictionary mapping gene names to the proteins they encode
# Conveniently enough, multiple occurrences of individual genes have
# already been collapsed and the proteins encoded by individual genes
# are stored in lists in the `protein_ids` column
gene_to_prot_dict = (
    pu_training_set_df.set_index("gene")["protein_ids"].to_dict()
)

In [8]:
# Extract the required columns/features
pheno = np.stack(pu_training_set_df["phenotype_vec"].to_numpy())
labels = pu_training_set_df["label"].to_numpy()

In [9]:
# For each gene, aggregate the corresponding PPI probability vectors
agg_ppi_prob_vector_dict = {}

for gene, prot_list in gene_to_prot_dict.items():
    ppi_prob_vector_list = [
        prot_to_ppi_prob_vector_dict[protein]
        for protein in prot_list
    ]
    agg_ppi_prob_vector_dict[gene] = np.max(
        ppi_prob_vector_list, axis=0
    )

In [10]:
# Save the aggregated PPI probability vectors by pickling them
with open("aggregated_PPI_probability_vectors_dict.pkl", "wb") as f:
    pickle.dump(agg_ppi_prob_vector_dict, f)

# Perform Prior Estimation

In [11]:
# Load the dictionary mapping genes to their aggregated PPI probability
# vector
with open("aggregated_PPI_probability_vectors_dict.pkl", "rb") as f:
    agg_ppi_prob_vector_dict = pickle.load(f)

In [12]:
# Now, the aggregated PPI probability vectors have to be stored in a
# NumPy array such that their order aligns with that of `pheno` and
# `labels`
ppi = np.stack(
    pu_training_set_df["gene"].apply(
        lambda gene: agg_ppi_prob_vector_dict[gene]
    ).to_numpy()
)

In [13]:
mean_prior = estimate_prior_elkan_noto_repeated(
    pheno,
    ppi,
    labels
)

π mean = 0.4846, std = 0.0042


# Prior Estimation for Different P:U Sampling Ratios

In [14]:
# Prior estimation is repeated for different P:U sampling ratios in
# order to investigate the stability of the estimated prior

# Bundle the preprocessing steps in a single function
def prior_estimation_preprocessing(training_set_df, ppi_df):
    # Create a dictionary mapping UniProt accessions to their
    # corresponding PPI probability vector of length 440
    prot_to_ppi_prob_vector_dict = {}

    # To this end, the `.groupby()` method is employed
    for protein, group in ppi_df.groupby(by="protein_1"):
        ppi_prob_vector = group["interaction_probability"].to_numpy()
        assert ppi_prob_vector.shape[0] == 440, (
            f"For protein {protein}, some PPI probabilities are missing!"
        )
        prot_to_ppi_prob_vector_dict[protein] = ppi_prob_vector
    
    # The probabilities must first be aggregated as some genes encode
    # multiple proteins
    # Create a dictionary mapping gene names to the proteins they encode
    # Conveniently enough, multiple occurrences of individual genes have
    # already been collaped and the proteins encoded by the individual
    # genes are stored as lists in the `protein_ids` column
    gene_to_prot_dict = (
        training_set_df.set_index("gene")["protein_ids"].to_dict()
    )
    
    # Extract the required columns/features
    pheno = np.stack(training_set_df["phenotype_vec"].to_numpy())
    labels = training_set_df["label"].to_numpy()

    # For each gene, aggregate the corresponding PPI probability vectors
    agg_ppi_prob_vector_dict = {}

    for gene, prot_list in gene_to_prot_dict.items():
        ppi_prob_vector_list = [
            prot_to_ppi_prob_vector_dict[protein]
            for protein in prot_list
        ]
        agg_ppi_prob_vector_dict[gene] = np.max(
            ppi_prob_vector_list, axis=0
        )
    
    # Now, the aggregated PPI probability vectors have to be stored in a
    # NumPy array such that their order aligns with that of `pheno` and
    # `labels`
    ppi = np.stack(
        training_set_df["gene"].apply(
            lambda gene: agg_ppi_prob_vector_dict[gene]
        ).to_numpy()
    )

    return pheno, ppi, labels

In [15]:
# Perform prior estimation for a P:U ratio of 1:10
# Load the corresponding training set
pu_training_set_1_10_df = pd.read_csv(
    "/Users/jacobanter/Documents/Code/VACV_screen/Positive_unlabeled_"
    "learning/Train_validation_test_split/PU_data_set_splits_PU_ratio_"
    "1_10/PU_training_set.tsv",
    sep="\t",
    converters={
        "phenotype_vec": ast.literal_eval,
        "protein_ids": ast.literal_eval
    }
)

pheno_1_10, ppi_1_10, labels_1_10 = prior_estimation_preprocessing(
    pu_training_set_1_10_df, ppi_preds_df
)

mean_prior_1_10 = estimate_prior_elkan_noto_repeated(
    pheno_1_10,
    ppi_1_10,
    labels_1_10
)

π mean = 0.3629, std = 0.0044


In [16]:
# Perform prior estimation for a P:U ratio of 1:20
# Load the corresponding training set
pu_training_set_1_20_df = pd.read_csv(
    "/Users/jacobanter/Documents/Code/VACV_screen/Positive_unlabeled_"
    "learning/Train_validation_test_split/PU_data_set_splits_PU_ratio_"
    "1_20/PU_training_set.tsv",
    sep="\t",
    converters={
        "phenotype_vec": ast.literal_eval,
        "protein_ids": ast.literal_eval
    }
)

pheno_1_20, ppi_1_20, labels_1_20 = prior_estimation_preprocessing(
    pu_training_set_1_20_df, ppi_preds_df
)

mean_prior_1_20 = estimate_prior_elkan_noto_repeated(
    pheno_1_20,
    ppi_1_20,
    labels_1_20
)

π mean = 0.2875, std = 0.0044
